# Building LLM

## Goal

This notebook is a hands-on journey to build a language model from scratch.

Each version introduces one new concept, allowing the model to evolve step by step while practicing language-model development.

---

## Version 15

Version 14 added supervised instruction tuning to the checkpointed and evaluated WikiText/BPE language model.

Version 15 keeps the same datasets, tokenizer, context length, two-block decoder structure, parameter count, training pipeline, checkpoint behavior, held-out evaluation, instruction-tuning stage and controlled generation.

The new concept is **explicit multi-head causal self-attention**.

The existing 16-dimensional attention space is divided into two independent 8-dimensional heads:

single 16-dimensional attention space  
→ two 8-dimensional attention heads  
→ independent causal attention distributions  
→ concatenated head outputs  
→ the same 8-dimensional model representation

Query, key, value and attention-output projection shapes remain unchanged, so the model still contains exactly 34,496 trainable parameters.

This version therefore isolates the effect of multiple attention heads without simultaneously changing model width, block count, optimizer, training budget or instruction-tuning procedure.

## 1. Imports and Configuration

Version 15 preserves the complete Version 14 experiment:

- 10,000 WikiText training rows, 1,000 validation rows and 1,000 test rows;
- a BPE vocabulary of 2,048 tokens;
- contexts of 16 tokens;
- 12,000 training sequences, 2,000 validation sequences and 2,000 test sequences;
- batch size 32 and learning rate 1.0;
- two decoder blocks;
- interruption at epoch 10 and a total base-training budget of 20 epochs;
- the same 192/32/32 Dolly instruction split and 8-epoch supervised fine-tuning stage;
- the same held-out evaluation and controlled-generation procedure.

The new architecture setting is `NUM_HEADS = 2`.

`ATTENTION_DIM` remains 16, so each head receives `HEAD_DIM = 8` dimensions.

Because the query, key, value and output projection matrices keep the same shapes as Version 14, introducing two heads does not change the number of trainable parameters.

In [1]:
import os
import shutil
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["RAYON_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import tensorflow as tf

from tokenizers import Tokenizer
from tokenizers.decoders import BPEDecoder
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

In [2]:
SEED = 42

TRAIN_ROWS = 10000
VALIDATION_ROWS = 1000
TEST_ROWS = 1000

VOCABULARY_SIZE = 2048
CONTEXT_LENGTH = 16

MAX_TRAIN_SEQUENCES = 12000
MAX_VALIDATION_SEQUENCES = 2000
MAX_TEST_SEQUENCES = 2000

BATCH_SIZE = 32
LEARNING_RATE = 1.0

INTERRUPT_AFTER_EPOCH = 10
TOTAL_EPOCHS = 20

EMBEDDING_DIM = 8
ATTENTION_DIM = 16
NUM_HEADS = 2
HEAD_DIM = ATTENTION_DIM // NUM_HEADS
FEED_FORWARD_DIM = 16
NUM_BLOCKS = 2
LAYER_NORM_EPSILON = 1e-5

assert ATTENTION_DIM % NUM_HEADS == 0

tf.keras.utils.set_random_seed(SEED)

INSTRUCTION_ROWS = 256
INSTRUCTION_TRAIN_ROWS = 192
INSTRUCTION_VALIDATION_ROWS = 32
INSTRUCTION_TEST_ROWS = 32

INSTRUCTION_BATCH_SIZE = 32
INSTRUCTION_LEARNING_RATE = 0.15
INSTRUCTION_EPOCHS = 8
MAX_WINDOWS_PER_INSTRUCTION = 2
INSTRUCTION_GENERATION_TOKENS = 16

In [3]:
DATASET_DIRECTORY = "/kaggle/input/datasets/lucalullo/wiki-text-103-raw-v1"

train_files = [
    os.path.join(DATASET_DIRECTORY, "train-00000-of-00002.parquet"),
    os.path.join(DATASET_DIRECTORY, "train-00001-of-00002.parquet"),
]

validation_file = os.path.join(DATASET_DIRECTORY, "validation-00000-of-00001.parquet")
test_file = os.path.join(DATASET_DIRECTORY, "test-00000-of-00001.parquet")

for path in train_files + [validation_file, test_file]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

print("Dataset files found:")
for path in train_files + [validation_file, test_file]:
    print(os.path.basename(path))

Dataset files found:
train-00000-of-00002.parquet
train-00001-of-00002.parquet
validation-00000-of-00001.parquet
test-00000-of-00001.parquet


## 2. Public Dataset

Version 15 keeps the **WikiText-103 Raw v1** base-training dataset used in Versions 11–14.

The official training, validation and test split boundaries remain unchanged.

The same reproducible subset is selected independently from each split:

- 10,000 training rows;
- 1,000 validation rows;
- 1,000 test rows.

Empty rows are removed before sampling. No new random train/validation/test split is created.

The tokenizer is still trained only on the selected WikiText training subset. The separate Dolly instruction dataset used later does not rebuild or modify the BPE vocabulary.

In [4]:
train_data = pd.concat(
    [pd.read_parquet(path, columns=["text"]) for path in train_files],
    ignore_index=True,
)

validation_data = pd.read_parquet(validation_file, columns=["text"])
test_data = pd.read_parquet(test_file, columns=["text"])

print("Original rows:")
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Original rows:
Train: 1801350
Validation: 3760
Test: 4358


In [5]:
def prepare_text_subset(data, number_of_rows, seed):
    cleaned = data.dropna(subset=["text"]).copy()
    cleaned = cleaned[cleaned["text"].str.strip().str.len() > 0]
    return cleaned.sample(n=min(number_of_rows, len(cleaned)), random_state=seed).reset_index(drop=True)


train_data = prepare_text_subset(train_data, TRAIN_ROWS, SEED)
validation_data = prepare_text_subset(validation_data, VALIDATION_ROWS, SEED)
test_data = prepare_text_subset(test_data, TEST_ROWS, SEED)

print("Experiment rows:")
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

print("\nExample training text:")
print(train_data.loc[0, "text"])

Experiment rows:
Train: 10000
Validation: 1000
Test: 1000

Example training text:
 In cooperation with various nonprofit organizations , since 2008 the company has organized the THSRC Smile Program , which offers minority groups and families the chance to experience the comfort and convenience of high @-@ speed rail rides . THSRC Smile Program gave its first ride on 23 January 2008 to children , social workers and parents from the Fushan branch of the Taiwan Fund for Children and Families . As of the end of 2011 , 12 @,@ 231 people in total , comprising 229 underprivileged groups and families , have participated in the THSRC Smile Program . 



## 3. Subword Tokenizer

Version 15 preserves the **Byte Pair Encoding (BPE)** tokenizer used since Version 11.

The tokenizer is trained only on the selected WikiText training subset with the same configuration:

- maximum vocabulary size 2,048;
- `[UNK]` unknown token;
- whitespace pre-tokenization;
- minimum frequency 2;
- `</w>` end-of-word suffix.

Dolly validation and test examples never influence tokenizer construction.

Version 15 changes how attention is organized inside the Transformer, not how text is represented.

In [6]:
tokenizer = Tokenizer(BPE(
    unk_token="[UNK]",
    end_of_word_suffix="</w>"
))

tokenizer.pre_tokenizer = Whitespace()
tokenizer.decoder = BPEDecoder(suffix="</w>")

trainer = BpeTrainer(
    vocab_size=VOCABULARY_SIZE,
    min_frequency=2,
    special_tokens=["[UNK]"],
    end_of_word_suffix="</w>"
)

tokenizer.train_from_iterator(train_data["text"], trainer=trainer)

vocabulary_size = tokenizer.get_vocab_size()

print("Vocabulary size:", vocabulary_size)




Vocabulary size: 2048


### Inspecting the Learned Tokens

The tokenizer can now represent text using variable-length subword units instead of individual characters.

Frequent words or word fragments may become single tokens, while less frequent words can be decomposed into multiple smaller tokens.

In [7]:
example_text = train_data.loc[0, "text"]
encoded_example = tokenizer.encode(example_text)

print("Text:")
print(example_text)

print("\nTokens:")
print(encoded_example.tokens[:30])

print("\nToken IDs:")
print(encoded_example.ids[:30])

print("\nNumber of characters:", len(example_text))
print("Number of tokens:", len(encoded_example.ids))

Text:
 In cooperation with various nonprofit organizations , since 2008 the company has organized the THSRC Smile Program , which offers minority groups and families the chance to experience the comfort and convenience of high @-@ speed rail rides . THSRC Smile Program gave its first ride on 23 January 2008 to children , social workers and parents from the Fushan branch of the Taiwan Fund for Children and Families . As of the end of 2011 , 12 @,@ 231 people in total , comprising 229 underprivileged groups and families , have participated in the THSRC Smile Program . 


Tokens:
['In</w>', 'co', 'oper', 'ation</w>', 'with</w>', 'vari', 'ous</w>', 'n', 'on', 'pro', 'f', 'it</w>', 'org', 'ani', 'z', 'ations</w>', ',</w>', 'since</w>', '200', '8</w>', 'the</w>', 'comp', 'any</w>', 'has</w>', 'org', 'ani', 'zed</w>', 'the</w>', 'T', 'H']

Token IDs:
[1146, 1228, 1730, 1103, 1066, 1919, 1272, 78, 999, 1082, 70, 1102, 1875, 1735, 90, 1434, 917, 1927, 1196, 707, 991, 1452, 1389, 1260, 1875, 173

## 4. Token Sequences

The decoder-only Transformer still learns next-token prediction at every context position.

Each WikiText row is encoded independently with the trained BPE tokenizer.

Token sequences are divided into fixed-length windows of `CONTEXT_LENGTH + 1` tokens. The first `CONTEXT_LENGTH` tokens form the input and the same sequence shifted by one position forms the target.

For example:

input tokens  → `[t0, t1, t2, ..., t15]`  
target tokens → `[t1, t2, t3, ..., t16]`

Unlike the character-level examples used in earlier versions, the windows now contain subword token identifiers.

Windows are created independently inside each text row so examples never cross row boundaries. A stride equal to `CONTEXT_LENGTH` avoids excessive overlap and keeps the experiment compact.

In [8]:
def create_token_sequences(texts, tokenizer, context_length):
    inputs, targets = [], []

    for text in texts:
        token_ids = tokenizer.encode(text).ids

        for start in range(0, len(token_ids) - context_length, context_length):
            sequence = token_ids[start:start + context_length + 1]

            if len(sequence) == context_length + 1:
                inputs.append(sequence[:-1])
                targets.append(sequence[1:])

    return tf.constant(inputs, dtype=tf.int32), tf.constant(targets, dtype=tf.int32)

In [9]:
train_inputs, train_targets = create_token_sequences(train_data["text"], tokenizer, CONTEXT_LENGTH)
validation_inputs, validation_targets = create_token_sequences(validation_data["text"], tokenizer, CONTEXT_LENGTH)
test_inputs, test_targets = create_token_sequences(test_data["text"], tokenizer, CONTEXT_LENGTH)

print("Train:", train_inputs.shape, train_targets.shape)
print("Validation:", validation_inputs.shape, validation_targets.shape)
print("Test:", test_inputs.shape, test_targets.shape)

print("\nFirst input IDs:")
print(train_inputs[0].numpy())

print("\nFirst target IDs:")
print(train_targets[0].numpy())

print("\nDecoded input:")
print(tokenizer.decode(train_inputs[0].numpy()))

Train: (96893, 16) (96893, 16)
Validation: (9764, 16) (9764, 16)
Test: (9572, 16) (9572, 16)

First input IDs:
[1146 1228 1730 1103 1066 1919 1272   78  999 1082   70 1102 1875 1735
   90 1434]

First target IDs:
[1228 1730 1103 1066 1919 1272   78  999 1082   70 1102 1875 1735   90
 1434  917]

Decoded input:
In cooperation with various nonprofit organizations


In [10]:
train_inputs = train_inputs[:MAX_TRAIN_SEQUENCES]
train_targets = train_targets[:MAX_TRAIN_SEQUENCES]

validation_inputs = validation_inputs[:MAX_VALIDATION_SEQUENCES]
validation_targets = validation_targets[:MAX_VALIDATION_SEQUENCES]

test_inputs = test_inputs[:MAX_TEST_SEQUENCES]
test_targets = test_targets[:MAX_TEST_SEQUENCES]

## 5. Explicit Multi-Head Decoder-Only Transformer

Version 15 changes only the internal organization of causal self-attention.

The existing query, key and value projections still map the 8-dimensional model representation into `ATTENTION_DIM = 16`.

Instead of treating those 16 dimensions as one attention space, they are reshaped into:

`NUM_HEADS = 2`  
×  
`HEAD_DIM = 8`

Each head computes its own causal scaled dot-product attention matrix.

The two head outputs are then concatenated back into 16 dimensions and passed through the same 16 → 8 attention-output projection used in Version 14.

Token embeddings, positional embeddings, two Transformer blocks, residual connections, LayerNorm, the 8 → 16 → 8 feed-forward network and final vocabulary projection remain unchanged.

Because all trainable matrix shapes remain unchanged, the model still contains exactly **34,496 trainable parameters**.

In [11]:
def layer_normalize(values, scale, shift):
    mean = tf.reduce_mean(values, axis=-1, keepdims=True)
    variance = tf.reduce_mean(tf.square(values - mean), axis=-1, keepdims=True)
    normalized = (values - mean) / tf.sqrt(variance + LAYER_NORM_EPSILON)
    return normalized * scale + shift


class TransformerBlock(tf.Module):
    def __init__(self, random_generator, name):
        super().__init__(name=name)

        self.query_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, ATTENTION_DIM), stddev=0.02), name="query_weights")
        self.key_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, ATTENTION_DIM), stddev=0.02), name="key_weights")
        self.value_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, ATTENTION_DIM), stddev=0.02), name="value_weights")
        self.attention_output_weights = tf.Variable(random_generator.normal((ATTENTION_DIM, EMBEDDING_DIM), stddev=0.02), name="attention_output_weights")

        self.layer_norm_1_scale = tf.Variable(tf.ones((EMBEDDING_DIM,)), name="layer_norm_1_scale")
        self.layer_norm_1_shift = tf.Variable(tf.zeros((EMBEDDING_DIM,)), name="layer_norm_1_shift")

        self.feed_forward_input_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, FEED_FORWARD_DIM), stddev=0.02), name="feed_forward_input_weights")
        self.feed_forward_output_weights = tf.Variable(random_generator.normal((FEED_FORWARD_DIM, EMBEDDING_DIM), stddev=0.02), name="feed_forward_output_weights")

        self.layer_norm_2_scale = tf.Variable(tf.ones((EMBEDDING_DIM,)), name="layer_norm_2_scale")
        self.layer_norm_2_shift = tf.Variable(tf.zeros((EMBEDDING_DIM,)), name="layer_norm_2_shift")

    @property
    def trainable_variables(self):
        return [
            self.query_weights,
            self.key_weights,
            self.value_weights,
            self.attention_output_weights,
            self.layer_norm_1_scale,
            self.layer_norm_1_shift,
            self.feed_forward_input_weights,
            self.feed_forward_output_weights,
            self.layer_norm_2_scale,
            self.layer_norm_2_shift,
        ]

    def split_heads(self, values):
        batch_size = tf.shape(values)[0]
        sequence_length = tf.shape(values)[1]
        values = tf.reshape(values, (batch_size, sequence_length, NUM_HEADS, HEAD_DIM))
        return tf.transpose(values, (0, 2, 1, 3))

    def __call__(self, values):
        queries = self.split_heads(tf.matmul(values, self.query_weights))
        keys = self.split_heads(tf.matmul(values, self.key_weights))
        values_projected = self.split_heads(tf.matmul(values, self.value_weights))

        scores = tf.matmul(queries, keys, transpose_b=True)
        scores = scores / tf.sqrt(tf.cast(HEAD_DIM, tf.float32))

        sequence_length = tf.shape(scores)[-1]
        causal_mask = tf.linalg.band_part(tf.ones((sequence_length, sequence_length)), -1, 0)
        masked_scores = tf.where(causal_mask == 1, scores, tf.constant(-1e9, dtype=tf.float32))

        attention_weights = tf.nn.softmax(masked_scores, axis=-1)
        head_outputs = tf.matmul(attention_weights, values_projected)

        head_outputs = tf.transpose(head_outputs, (0, 2, 1, 3))
        batch_size = tf.shape(head_outputs)[0]
        concatenated = tf.reshape(head_outputs, (batch_size, sequence_length, ATTENTION_DIM))

        attention_output = tf.matmul(concatenated, self.attention_output_weights)

        values = layer_normalize(values + attention_output, self.layer_norm_1_scale, self.layer_norm_1_shift,)

        feed_forward = tf.nn.relu(tf.matmul(values, self.feed_forward_input_weights))
        feed_forward = tf.matmul(feed_forward, self.feed_forward_output_weights)

        output = layer_normalize(values + feed_forward, self.layer_norm_2_scale, self.layer_norm_2_shift,)

        return output, attention_weights


class DecoderOnlyLanguageModel(tf.Module):
    def __init__(self, vocabulary_size, seed=42, name="decoder_only_language_model"):
        super().__init__(name=name)
        random_generator = tf.random.Generator.from_seed(seed)

        self.token_embeddings = tf.Variable(
            random_generator.normal((vocabulary_size, EMBEDDING_DIM), stddev=0.02),
            name="token_embeddings",
        )

        self.positional_embeddings = tf.Variable(
            random_generator.normal((CONTEXT_LENGTH, EMBEDDING_DIM), stddev=0.02),
            name="positional_embeddings",
        )

        self.blocks = [
            TransformerBlock(random_generator, name=f"transformer_block_{index + 1}")
            for index in range(NUM_BLOCKS)
        ]

        self.output_weights = tf.Variable(
            random_generator.normal((EMBEDDING_DIM, vocabulary_size), stddev=0.02),
            name="output_weights",
        )

    @property
    def trainable_variables(self):
        variables = [self.token_embeddings, self.positional_embeddings]

        for block in self.blocks:
            variables.extend(block.trainable_variables)

        variables.append(self.output_weights)
        return variables

    def __call__(self, input_ids):
        token_values = tf.gather(self.token_embeddings, input_ids)
        values = token_values + self.positional_embeddings
        attention_weights = []

        for block in self.blocks:
            values, block_attention = block(values)
            attention_weights.append(block_attention)

        logits = tf.matmul(values, self.output_weights)
        return values, attention_weights, logits

In [12]:
vocabulary_size = tokenizer.get_vocab_size()

model = DecoderOnlyLanguageModel(vocabulary_size=vocabulary_size, seed=SEED)

print("Tokenizer vocabulary size:", vocabulary_size)
print("Token embedding shape:", model.token_embeddings.shape)

Tokenizer vocabulary size: 2048
Token embedding shape: (2048, 8)


### Model Structure and Trainable Parameters

The tokenizer vocabulary remains 2,048 tokens and every model context contains 16 token positions.

Two independent Transformer blocks still process token and positional embeddings before the final vocabulary projection.

Inside each block, the 16-dimensional attention projection is now divided into two independent 8-dimensional heads.

Each block therefore returns an attention tensor with shape:

`(batch, 2, 16, 16)`

The projection dimensions remain unchanged, so the parameter count stays exactly 34,496. This isolates multi-head attention from changes in model size.

In [13]:
example_input = train_inputs[:1]

model_output, attention_weights, logits = model(example_input)

total_parameters = sum(
    int(np.prod(variable.shape))
    for variable in model.trainable_variables
)

print("Vocabulary size:", vocabulary_size)
print("Number of Transformer blocks:", len(model.blocks))
print("Number of attention heads:", NUM_HEADS)
print("Head dimension:", HEAD_DIM)
print("Number of trainable variables:", len(model.trainable_variables))

print("\nInput shape:", example_input.shape)
print("Model output shape:", model_output.shape)
print("Logits shape:", logits.shape)

for block_index, block_attention in enumerate(attention_weights, start=1):
    print(f"Block {block_index} attention shape:", block_attention.shape)

    for head_index in range(NUM_HEADS):
        final_row = block_attention[0, head_index, -1].numpy()
        print(f"  Head {head_index + 1} final-position sum: {final_row.sum():.6f}")

print("\nTotal trainable parameters:", total_parameters)

Vocabulary size: 2048
Number of Transformer blocks: 2
Number of attention heads: 2
Head dimension: 8
Number of trainable variables: 23

Input shape: (1, 16)
Model output shape: (1, 16, 8)
Logits shape: (1, 16, 2048)
Block 1 attention shape: (1, 2, 16, 16)
  Head 1 final-position sum: 1.000000
  Head 2 final-position sum: 1.000000
Block 2 attention shape: (1, 2, 16, 16)
  Head 1 final-position sum: 1.000000
  Head 2 final-position sum: 1.000000

Total trainable parameters: 34496


## 6. Base Training Pipeline

Version 15 preserves the Version 14 base-training pipeline.

The next-token objective, sparse cross-entropy, manual gradient updates, batch size, learning rate, deterministic epoch-specific shuffling and validation procedure remain unchanged.

The checkpoint demonstration is also preserved so the instruction-tuning stage still starts from a validation-selected base model.

The only change is that every Transformer block now computes two causal attention heads instead of one attention distribution.

In [14]:
def make_dataset(inputs, targets, batch_size, shuffle_seed=None):
    dataset = tf.data.Dataset.from_tensor_slices((inputs, targets))

    if shuffle_seed is not None:
        dataset = dataset.shuffle(buffer_size=len(inputs), seed=shuffle_seed, reshuffle_each_iteration=False)

    return dataset.batch(batch_size)


validation_dataset = make_dataset(validation_inputs, validation_targets, BATCH_SIZE)
test_dataset = make_dataset(test_inputs, test_targets, BATCH_SIZE)

print("Training sequences:", len(train_inputs))
print("Validation sequences:", len(validation_inputs))
print("Test sequences:", len(test_inputs))
print("Batches per training epoch:", int(np.ceil(len(train_inputs) / BATCH_SIZE)))

Training sequences: 12000
Validation sequences: 2000
Test sequences: 2000
Batches per training epoch: 375


In [15]:
def compute_batch_loss(model, inputs, targets):
    _, _, logits = model(inputs)

    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=targets, logits=logits)

    return tf.reduce_mean(losses)

In [16]:
def evaluate_model(model, dataset):
    total_loss = 0.0
    total_tokens = 0

    for batch_inputs, batch_targets in dataset:
        batch_loss = compute_batch_loss(model, batch_inputs, batch_targets)
        batch_tokens = int(tf.size(batch_targets))

        total_loss += float(batch_loss) * batch_tokens
        total_tokens += batch_tokens

    return total_loss / total_tokens

### Gradient Descent

Each mini-batch performs one gradient-descent update.

TensorFlow's `GradientTape` records the operations used to compute the batch loss and obtains the gradient of that loss with respect to every trainable variable.

The parameters are then updated explicitly using the configured learning rate.

The mean training loss is accumulated from the mini-batches processed during each epoch, while validation loss is computed separately without modifying the model parameters.

Token embedding lookups can produce sparse `IndexedSlices` gradients. These are updated with `scatter_sub`, while dense gradients use the standard `assign_sub` update.

In [17]:
def apply_gradient(variable, gradient, learning_rate):
    if gradient is None:
        return

    if isinstance(gradient, tf.IndexedSlices):
        update = tf.IndexedSlices(
            values=learning_rate * gradient.values,
            indices=gradient.indices,
            dense_shape=gradient.dense_shape
        )
        variable.scatter_sub(update)

    else:
        variable.assign_sub(learning_rate * gradient)

In [18]:
def train_one_epoch(model, inputs, targets, epoch, learning_rate):
    training_dataset = make_dataset(
        inputs,
        targets,
        BATCH_SIZE,
        shuffle_seed=SEED + epoch,
    )

    total_loss = 0.0
    total_tokens = 0

    for batch_inputs, batch_targets in training_dataset:
        with tf.GradientTape() as tape:
            batch_loss = compute_batch_loss(model, batch_inputs, batch_targets)

        gradients = tape.gradient(batch_loss, model.trainable_variables)

        for variable, gradient in zip(model.trainable_variables, gradients):
            apply_gradient(variable, gradient, learning_rate)

        batch_tokens = int(tf.size(batch_targets))
        total_loss += float(batch_loss) * batch_tokens
        total_tokens += batch_tokens

    return total_loss / total_tokens

## 7. Base Training State and Checkpoints

Version 15 keeps the minimum checkpoint state introduced in Version 12:

- `completed_epoch`;
- `best_epoch`;
- `best_validation_loss`.

The **last checkpoint** answers where base training can resume.

The **best checkpoint** stores the base model selected by validation loss.

The runtime directory starts clean on every execution. Base training first reaches epoch 10, restores LAST into a fresh multi-head model, resumes through epoch 20 and then restores BEST into `final_model`.

In [19]:
RUNTIME_DIRECTORY = Path("/kaggle/working/runtime-v15")
BASE_RUNTIME_DIRECTORY = RUNTIME_DIRECTORY / "base"
INSTRUCTION_RUNTIME_DIRECTORY = RUNTIME_DIRECTORY / "instruction"

if RUNTIME_DIRECTORY.exists():
    shutil.rmtree(RUNTIME_DIRECTORY)

BASE_RUNTIME_DIRECTORY.mkdir(parents=True, exist_ok=True)
INSTRUCTION_RUNTIME_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Runtime checkpoints:", RUNTIME_DIRECTORY)

Runtime checkpoints: /kaggle/working/runtime-v15


In [20]:
def build_checkpoint_managers(
    model,
    completed_epoch,
    best_epoch,
    best_validation_loss,
    runtime_directory,
):
    last_checkpoint = tf.train.Checkpoint(
        model=model,
        completed_epoch=completed_epoch,
        best_epoch=best_epoch,
        best_validation_loss=best_validation_loss,
    )

    best_checkpoint = tf.train.Checkpoint(
        model=model,
        best_epoch=best_epoch,
        best_validation_loss=best_validation_loss,
    )

    last_manager = tf.train.CheckpointManager(
        last_checkpoint,
        str(runtime_directory / "last"),
        max_to_keep=1,
    )

    best_manager = tf.train.CheckpointManager(
        best_checkpoint,
        str(runtime_directory / "best"),
        max_to_keep=1,
    )

    return last_checkpoint, best_checkpoint, last_manager, best_manager

In [21]:
def run_training_pipeline(
    model,
    completed_epoch,
    best_epoch,
    best_validation_loss,
    train_inputs,
    train_targets,
    validation_dataset,
    final_epoch,
    learning_rate,
    last_manager,
    best_manager,
):
    start_epoch = int(completed_epoch.numpy()) + 1

    for epoch in range(start_epoch, final_epoch + 1):
        training_loss = train_one_epoch(
            model,
            train_inputs,
            train_targets,
            epoch,
            learning_rate,
        )

        validation_loss = evaluate_model(model, validation_dataset)

        completed_epoch.assign(epoch)

        if validation_loss < float(best_validation_loss.numpy()):
            best_validation_loss.assign(validation_loss)
            best_epoch.assign(epoch)
            best_manager.save(checkpoint_number=epoch)

        last_manager.save(checkpoint_number=epoch)

        if epoch == 1 or epoch % 10 == 0 or epoch == final_epoch:
            print(
                f"Epoch {epoch:3d} | "
                f"Train loss: {training_loss:.6f} | "
                f"Validation loss: {validation_loss:.6f}"
            )

In [22]:
completed_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
best_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
best_validation_loss = tf.Variable(np.inf, dtype=tf.float32, trainable=False)

last_checkpoint, best_checkpoint, last_manager, best_manager = build_checkpoint_managers(
    model,
    completed_epoch,
    best_epoch,
    best_validation_loss,
    BASE_RUNTIME_DIRECTORY,
)

initial_train_dataset = make_dataset(train_inputs, train_targets, BATCH_SIZE)

initial_train_loss = evaluate_model(model, initial_train_dataset)
initial_validation_loss = evaluate_model(model, validation_dataset)
random_baseline = float(np.log(vocabulary_size))

print(f"Initial training loss: {initial_train_loss:.6f}")
print(f"Initial validation loss: {initial_validation_loss:.6f}")
print(f"Random baseline: {random_baseline:.6f}")

Initial training loss: 7.626327
Initial validation loss: 7.625874
Random baseline: 7.624619


## 8. Simulated Base Interruption, Restoration and Resume

The checkpoint behavior introduced in Version 12 remains part of the base pipeline.

Training first reaches epoch 10 and writes LAST and BEST checkpoints.

A fresh multi-head model restores LAST, recovers `completed_epoch = 10`, and resumes from epoch 11 through epoch 20.

Validation continues to update BEST whenever a lower validation loss is observed.

In [23]:
run_training_pipeline(
    model,
    completed_epoch,
    best_epoch,
    best_validation_loss,
    train_inputs,
    train_targets,
    validation_dataset,
    INTERRUPT_AFTER_EPOCH,
    LEARNING_RATE,
    last_manager,
    best_manager,
)

print("\nInterrupted after epoch:", int(completed_epoch.numpy()))
print("Last checkpoint:", last_manager.latest_checkpoint)
print("Best checkpoint:", best_manager.latest_checkpoint)

Epoch   1 | Train loss: 6.021490 | Validation loss: 5.510716
Epoch  10 | Train loss: 5.071039 | Validation loss: 5.113590

Interrupted after epoch: 10
Last checkpoint: /kaggle/working/runtime-v15/base/last/ckpt-10
Best checkpoint: /kaggle/working/runtime-v15/base/best/ckpt-10


In [24]:
resumed_model = DecoderOnlyLanguageModel(vocabulary_size=vocabulary_size, seed=SEED)
_ = resumed_model(example_input)

resumed_completed_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
resumed_best_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
resumed_best_validation_loss = tf.Variable(np.inf, dtype=tf.float32, trainable=False)

(
    resumed_last_checkpoint,
    resumed_best_checkpoint,
    resumed_last_manager,
    resumed_best_manager,
) = build_checkpoint_managers(
    resumed_model,
    resumed_completed_epoch,
    resumed_best_epoch,
    resumed_best_validation_loss,
    BASE_RUNTIME_DIRECTORY,
)

if resumed_last_manager.latest_checkpoint is None:
    raise RuntimeError("Last checkpoint not found.")

resumed_last_checkpoint.restore(resumed_last_manager.latest_checkpoint).assert_existing_objects_matched()

restored_epoch = int(resumed_completed_epoch.numpy())

print("Restored completed epoch:", restored_epoch)
print("Resume begins at epoch:", restored_epoch + 1)

Restored completed epoch: 10
Resume begins at epoch: 11


In [25]:
run_training_pipeline(
    resumed_model,
    resumed_completed_epoch,
    resumed_best_epoch,
    resumed_best_validation_loss,
    train_inputs,
    train_targets,
    validation_dataset,
    TOTAL_EPOCHS,
    LEARNING_RATE,
    resumed_last_manager,
    resumed_best_manager,
)

print("\nCompleted epoch:", int(resumed_completed_epoch.numpy()))
print("Best validation epoch:", int(resumed_best_epoch.numpy()))
print("Best validation loss:", f"{float(resumed_best_validation_loss.numpy()):.6f}")

Epoch  20 | Train loss: 5.011402 | Validation loss: 5.077443

Completed epoch: 20
Best validation epoch: 18
Best validation loss: 5.075184


## 9. Restore and Evaluate the Best Base Model

After the 20-epoch base-training budget is complete, a fresh `final_model` is restored from the checkpoint with the lowest validation loss.

Training and validation losses are recomputed from this restored multi-head model.

The held-out evaluation introduced in Version 13 is preserved: the WikiText test split is measured with cross-entropy, perplexity, top-1 token accuracy and top-5 token accuracy.

This validation-selected and evaluated base model becomes the fixed starting point for the inherited supervised instruction-tuning stage.

In [26]:
final_model = DecoderOnlyLanguageModel(vocabulary_size=vocabulary_size, seed=SEED)
_ = final_model(example_input)

selected_best_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
selected_best_validation_loss = tf.Variable(np.inf, dtype=tf.float32, trainable=False)

final_best_checkpoint = tf.train.Checkpoint(
    model=final_model,
    best_epoch=selected_best_epoch,
    best_validation_loss=selected_best_validation_loss,
)

if resumed_best_manager.latest_checkpoint is None:
    raise RuntimeError("Best checkpoint not found.")

final_best_checkpoint.restore(resumed_best_manager.latest_checkpoint).assert_existing_objects_matched()

final_train_loss = evaluate_model(final_model, make_dataset(train_inputs, train_targets, BATCH_SIZE))
final_validation_loss = evaluate_model(final_model, validation_dataset)

print("Selected best epoch:", int(selected_best_epoch.numpy()))
print(f"Final training loss: {final_train_loss:.6f}")
print(f"Final validation loss: {final_validation_loss:.6f}")
print(f"Random baseline: {random_baseline:.6f}")

Selected best epoch: 18
Final training loss: 4.982092
Final validation loss: 5.075184
Random baseline: 7.624619


In [27]:
def compute_evaluation_metrics(model, dataset):
    total_negative_log_likelihood = 0.0
    total_tokens = 0
    top_1_correct = 0
    top_5_correct = 0

    for batch_inputs, batch_targets in dataset:
        _, _, batch_logits = model(batch_inputs)

        token_losses = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=batch_targets, logits=batch_logits)
        flattened_logits = tf.reshape(batch_logits, (-1, vocabulary_size))
        flattened_targets = tf.reshape(batch_targets, (-1,))

        top_1_predictions = tf.argmax(flattened_logits, axis=-1, output_type=tf.int32)
        top_1_correct += int(tf.reduce_sum(tf.cast(top_1_predictions == flattened_targets, tf.int32)))

        top_5_correct += int(tf.reduce_sum(tf.cast(
            tf.math.in_top_k(targets=flattened_targets, predictions=flattened_logits, k=5),
            tf.int32,
        )))

        total_negative_log_likelihood += float(tf.reduce_sum(token_losses))
        total_tokens += int(tf.size(batch_targets))

    cross_entropy = total_negative_log_likelihood / total_tokens

    return {
        "cross_entropy": cross_entropy,
        "perplexity": float(np.exp(cross_entropy)),
        "top_1_accuracy": top_1_correct / total_tokens,
        "top_5_accuracy": top_5_correct / total_tokens,
        "tokens": total_tokens,
    }

In [28]:
base_test_metrics = compute_evaluation_metrics(final_model, test_dataset)
base_test_loss = base_test_metrics["cross_entropy"]
random_perplexity = float(np.exp(random_baseline))

print("Held-out test metrics:")
print(f"Cross-entropy: {base_test_metrics['cross_entropy']:.6f}")
print(f"Perplexity:    {base_test_metrics['perplexity']:.3f}")
print(f"Top-1 accuracy: {base_test_metrics['top_1_accuracy']:.4f}")
print(f"Top-5 accuracy: {base_test_metrics['top_5_accuracy']:.4f}")
print(f"Evaluated tokens: {base_test_metrics['tokens']}")

print(f"\nUniform baseline cross-entropy: {random_baseline:.6f}")
print(f"Uniform baseline perplexity: {random_perplexity:.1f}")

Held-out test metrics:
Cross-entropy: 5.098042
Perplexity:    163.701
Top-1 accuracy: 0.0986
Top-5 accuracy: 0.2401
Evaluated tokens: 32000

Uniform baseline cross-entropy: 7.624619
Uniform baseline perplexity: 2048.0


## 10. External Instruction Dataset

Version 15 preserves the **Databricks Dolly 15K** instruction-tuning stage introduced in Version 14.

The same small deterministic subset is used:

- 192 training examples;
- 32 validation examples;
- 32 test examples.

Rows with empty instructions or responses are removed and duplicate instruction/context/response triples are discarded before sampling.

The split is created **before** token windows are constructed, keeping instruction training, validation and test examples separate.

The attached Kaggle dataset remains `lucalullo/databricks-dolly-15k-official-dataset-mirror`, containing `databricks-dolly-15k.jsonl`.

No download or fallback instruction corpus is used.

In [29]:
INSTRUCTION_DATASET_DIRECTORY = "/kaggle/input/datasets/lucalullo/databricks-dolly-15k-official-dataset-mirror"
INSTRUCTION_DATASET_FILE = os.path.join(
    INSTRUCTION_DATASET_DIRECTORY,
    "databricks-dolly-15k.jsonl",
)

if not os.path.exists(INSTRUCTION_DATASET_FILE):
    raise FileNotFoundError(
        "Databricks Dolly 15K is required for Version 15.\n"
        f"Expected file: {INSTRUCTION_DATASET_FILE}\n"
        "Attach the Kaggle dataset "
        "lucalullo/databricks-dolly-15k-official-dataset-mirror."
    )

instruction_data = pd.read_json(INSTRUCTION_DATASET_FILE, lines=True)

required_columns = {"instruction", "context", "response"}
missing_columns = required_columns - set(instruction_data.columns)

if missing_columns:
    raise ValueError(
        "The Dolly JSONL file is missing required fields: "
        + ", ".join(sorted(missing_columns))
    )

instruction_data = instruction_data.copy()
instruction_data["source_index"] = instruction_data.index.astype(int)

for column in ("instruction", "context", "response"):
    instruction_data[column] = instruction_data[column].fillna("").astype(str)

instruction_data = instruction_data[instruction_data["instruction"].str.strip().str.len() > 0]

instruction_data = instruction_data[instruction_data["response"].str.strip().str.len() > 0]

instruction_data = instruction_data.drop_duplicates(subset=["instruction", "context", "response"])

if len(instruction_data) < INSTRUCTION_ROWS:
    raise ValueError(
        f"Dolly contains {len(instruction_data)} usable unique rows; "
        f"{INSTRUCTION_ROWS} are required."
    )

selected_instruction_data = instruction_data.sample(n=INSTRUCTION_ROWS, random_state=SEED).reset_index(drop=True)

instruction_train_data = selected_instruction_data.iloc[:INSTRUCTION_TRAIN_ROWS].reset_index(drop=True)

instruction_validation_data = selected_instruction_data.iloc[
    INSTRUCTION_TRAIN_ROWS:
    INSTRUCTION_TRAIN_ROWS + INSTRUCTION_VALIDATION_ROWS
].reset_index(drop=True)

instruction_test_data = selected_instruction_data.iloc[INSTRUCTION_TRAIN_ROWS + INSTRUCTION_VALIDATION_ROWS:].reset_index(drop=True)

print("Instruction rows:")
print("Train:", len(instruction_train_data))
print("Validation:", len(instruction_validation_data))
print("Test:", len(instruction_test_data))

Instruction rows:
Train: 192
Validation: 32
Test: 32


## 11. Transparent Prompt Formatting

Every instruction example is converted into a simple visible template:

`Instruction:`  
`Context:` *(only when present)*  
`Response:`

The formatting is deliberately explicit rather than hidden inside a chat template.

The prompt is presented to the causal model as normal input tokens. The reference response is appended only when supervised training sequences are constructed.

In [30]:
def format_instruction_prompt(example):
    prompt = f"Instruction:\n{example['instruction'].strip()}"

    if example["context"].strip():
        prompt += f"\n\nContext:\n{example['context'].strip()}"

    return prompt + "\n\nResponse:\n"


def format_instruction_example(example):
    return format_instruction_prompt(example) + example["response"].strip()


formatted_example = instruction_train_data.iloc[0].to_dict()

print("Formatted prompt:")
print(format_instruction_prompt(formatted_example))

print("\nReference response:")
print(formatted_example["response"].strip())

Formatted prompt:
Instruction:
Given a reference text about Pwojè Pyebwa, who funded the project?

Context:
Pwojè Pyebwa (Tree Project) is a tree-planting project in Haiti. It was designed, implemented, and initially run by cultural anthropologists. Different from reforestation projects, Pwojè Pyebwa promoted agroforestry—the strategic mixing of crops, trees, and animals. The agroforestry project in Haiti project was funded by the United States Agency for International Development USAID, but implemented through the Pan American Development Organization (PADF) and CARE. It was originally called the Agroforestry Outreach Project (AOP). The original project spanned a decade (1981–1991), but policies from Pwojè Pyebwa continued to direct PADF tree-planting through a second decade (1992 to 2000).

The project was executed throughout the country. Based on Haitian farmers’ voluntary and widespread participation, the project has been deemed a success by many. Pwojè Pyebwa distributed free tree

## 12. Response-Focused Training Sequences

Instruction tuning still uses causal next-token prediction, but prompt and response tokens have different training roles.

The prompt remains visible to causal attention so it can condition the response.

A binary **response mask** selects which target positions contribute to the supervised loss:

- `0` for prompt targets;
- `1` for response targets.

The masked loss is:

`sum(token_loss × response_mask) / sum(response_mask)`

This keeps the model architecture unchanged while focusing supervised updates on the desired response text.

Because the inherited context length is only 16 tokens, each instruction example contributes at most two response-containing windows in this educational experiment.

In [31]:
def create_instruction_sequences(data, tokenizer, context_length, max_windows_per_example):
    inputs, targets, masks, source_indices = [], [], [], []

    for _, example_row in data.iterrows():
        example = example_row.to_dict()
        prompt_ids = tokenizer.encode(format_instruction_prompt(example)).ids
        full_ids = tokenizer.encode(format_instruction_example(example)).ids

        if full_ids[:len(prompt_ids)] != prompt_ids:
            raise ValueError("Prompt tokenization is not a prefix of the full example.")

        response_start = len(prompt_ids)
        first_start = max(0, response_start - context_length)
        windows_added = 0

        for start in range(first_start, len(full_ids) - context_length, context_length):
            sequence = full_ids[start:start + context_length + 1]

            if len(sequence) != context_length + 1:
                continue

            target_positions = np.arange(start + 1, start + context_length + 1)
            response_mask = (target_positions >= response_start).astype(np.float32)

            if response_mask.sum() == 0:
                continue

            inputs.append(sequence[:-1])
            targets.append(sequence[1:])
            masks.append(response_mask)
            source_indices.append(int(example["source_index"]))

            windows_added += 1

            if windows_added == max_windows_per_example:
                break

    return (
        tf.constant(inputs, dtype=tf.int32),
        tf.constant(targets, dtype=tf.int32),
        tf.constant(masks, dtype=tf.float32),
        np.asarray(source_indices, dtype=np.int32),
    )


(
    instruction_train_inputs,
    instruction_train_targets,
    instruction_train_masks,
    instruction_train_sources,
) = create_instruction_sequences(
    instruction_train_data,
    tokenizer,
    CONTEXT_LENGTH,
    MAX_WINDOWS_PER_INSTRUCTION,
)

(
    instruction_validation_inputs,
    instruction_validation_targets,
    instruction_validation_masks,
    instruction_validation_sources,
) = create_instruction_sequences(
    instruction_validation_data,
    tokenizer,
    CONTEXT_LENGTH,
    MAX_WINDOWS_PER_INSTRUCTION,
)

(
    instruction_test_inputs,
    instruction_test_targets,
    instruction_test_masks,
    instruction_test_sources,
) = create_instruction_sequences(
    instruction_test_data,
    tokenizer,
    CONTEXT_LENGTH,
    MAX_WINDOWS_PER_INSTRUCTION,
)

print("Instruction sequence shapes:")
print("Train:", instruction_train_inputs.shape)
print("Validation:", instruction_validation_inputs.shape)
print("Test:", instruction_test_inputs.shape)

inspection_mask = instruction_train_masks[0].numpy()

print("\nFirst response mask:")
print(inspection_mask)
print("Response targets:", int(inspection_mask.sum()))

Instruction sequence shapes:
Train: (361, 16)
Validation: (60, 16)
Test: (61, 16)

First response mask:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
Response targets: 1


## 13. Masked Supervised Fine-Tuning

The supervised instruction-tuning procedure from Version 14 is preserved.

A new multi-head model instance is initialized from the validation-selected base model.

The same 34,496 parameters are then updated with the masked instruction objective.

Instruction validation loss is measured after every fine-tuning epoch. A **best instruction checkpoint** is saved only when that validation loss improves.

Version 15 does not add another optimizer, checkpoint contract or resume mechanism. The new concept remains isolated inside the attention computation.

The compact 8-epoch instruction run still prints epochs 1, 4 and 8.

In [32]:
def make_instruction_dataset(inputs, targets, masks, batch_size, shuffle_seed=None):
    dataset = tf.data.Dataset.from_tensor_slices((inputs, targets, masks))

    if shuffle_seed is not None:
        dataset = dataset.shuffle(
            buffer_size=len(inputs),
            seed=shuffle_seed,
            reshuffle_each_iteration=False,
        )

    return dataset.batch(batch_size)


def compute_masked_batch_loss(model, inputs, targets, response_mask):
    _, _, logits = model(inputs)

    token_losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=targets,
        logits=logits,
    )

    masked_losses = token_losses * response_mask
    return tf.reduce_sum(masked_losses) / tf.reduce_sum(response_mask)


def evaluate_masked_loss(model, dataset):
    total_loss = 0.0
    total_response_tokens = 0.0

    for batch_inputs, batch_targets, batch_mask in dataset:
        batch_loss = compute_masked_batch_loss(
            model,
            batch_inputs,
            batch_targets,
            batch_mask,
        )

        response_tokens = float(tf.reduce_sum(batch_mask))
        total_loss += float(batch_loss) * response_tokens
        total_response_tokens += response_tokens

    return total_loss / total_response_tokens


instruction_validation_dataset = make_instruction_dataset(
    instruction_validation_inputs,
    instruction_validation_targets,
    instruction_validation_masks,
    INSTRUCTION_BATCH_SIZE,
)

instruction_test_dataset = make_instruction_dataset(
    instruction_test_inputs,
    instruction_test_targets,
    instruction_test_masks,
    INSTRUCTION_BATCH_SIZE,
)

instruction_model = DecoderOnlyLanguageModel(
    vocabulary_size=vocabulary_size,
    seed=SEED,
)

_ = instruction_model(example_input)

for base_variable, instruction_variable in zip(
    final_model.trainable_variables,
    instruction_model.trainable_variables,
):
    instruction_variable.assign(base_variable)

instruction_initial_parameters = [
    variable.numpy().copy()
    for variable in instruction_model.trainable_variables
]

instruction_start_matches_base = all(
    np.array_equal(base_variable.numpy(), instruction_variable.numpy())
    for base_variable, instruction_variable in zip(
        final_model.trainable_variables,
        instruction_model.trainable_variables,
    )
)

initial_instruction_train_loss = evaluate_masked_loss(
    instruction_model,
    make_instruction_dataset(
        instruction_train_inputs,
        instruction_train_targets,
        instruction_train_masks,
        INSTRUCTION_BATCH_SIZE,
    ),
)

initial_instruction_validation_loss = evaluate_masked_loss(
    instruction_model,
    instruction_validation_dataset,
)

instruction_best_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
instruction_best_validation_loss = tf.Variable(np.inf, dtype=tf.float32, trainable=False)

instruction_best_checkpoint = tf.train.Checkpoint(
    model=instruction_model,
    best_epoch=instruction_best_epoch,
    best_validation_loss=instruction_best_validation_loss,
)

instruction_best_manager = tf.train.CheckpointManager(
    instruction_best_checkpoint,
    str(INSTRUCTION_RUNTIME_DIRECTORY / "best"),
    max_to_keep=1,
)

print("Instruction model starts from base BEST:", instruction_start_matches_base)
print(f"Initial masked training loss: {initial_instruction_train_loss:.6f}")
print(f"Initial masked validation loss: {initial_instruction_validation_loss:.6f}")

Instruction model starts from base BEST: True
Initial masked training loss: 5.273911
Initial masked validation loss: 5.297913


In [33]:
def train_instruction_epoch(model, inputs, targets, masks, epoch, learning_rate):
    dataset = make_instruction_dataset(
        inputs,
        targets,
        masks,
        INSTRUCTION_BATCH_SIZE,
        shuffle_seed=SEED + 1000 + epoch,
    )

    total_loss = 0.0
    total_response_tokens = 0.0

    for batch_inputs, batch_targets, batch_mask in dataset:
        with tf.GradientTape() as tape:
            batch_loss = compute_masked_batch_loss(model, batch_inputs, batch_targets, batch_mask)

        gradients = tape.gradient(batch_loss, model.trainable_variables)

        for variable, gradient in zip(model.trainable_variables, gradients):
            apply_gradient(variable, gradient, learning_rate)

        response_tokens = float(tf.reduce_sum(batch_mask))
        total_loss += float(batch_loss) * response_tokens
        total_response_tokens += response_tokens

    return total_loss / total_response_tokens


for epoch in range(1, INSTRUCTION_EPOCHS + 1):
    instruction_training_loss = train_instruction_epoch(
        instruction_model,
        instruction_train_inputs,
        instruction_train_targets,
        instruction_train_masks,
        epoch,
        INSTRUCTION_LEARNING_RATE,
    )

    instruction_validation_loss = evaluate_masked_loss(instruction_model, instruction_validation_dataset)

    if instruction_validation_loss < float(instruction_best_validation_loss.numpy()):
        instruction_best_validation_loss.assign(instruction_validation_loss)
        instruction_best_epoch.assign(epoch)
        instruction_best_manager.save(checkpoint_number=epoch)

    if epoch == 1 or epoch % 4 == 0 or epoch == INSTRUCTION_EPOCHS:
        print(
            f"Instruction epoch {epoch:2d} | "
            f"Train masked loss: {instruction_training_loss:.6f} | "
            f"Validation masked loss: {instruction_validation_loss:.6f}"
        )

instruction_parameters_updated = any(
    not np.array_equal(before, variable.numpy())
    for before, variable in zip(
        instruction_initial_parameters,
        instruction_model.trainable_variables,
    )
)

print("\nBest instruction epoch:", int(instruction_best_epoch.numpy()))
print(
    "Best instruction validation loss:",
    f"{float(instruction_best_validation_loss.numpy()):.6f}",
)

Instruction epoch  1 | Train masked loss: 5.226303 | Validation masked loss: 5.230621
Instruction epoch  4 | Train masked loss: 4.985866 | Validation masked loss: 5.187424
Instruction epoch  8 | Train masked loss: 4.817715 | Validation masked loss: 5.183962

Best instruction epoch: 5
Best instruction validation loss: 5.180870


## 14. Restore the Best Instruction-Tuned Model

A fresh model is restored from the instruction checkpoint with the lowest masked validation loss.

Only after this model has been selected are the held-out Dolly test sequences evaluated.

The final instruction metrics therefore preserve the same train/validation/test discipline used throughout the project.

In [34]:
instruction_tuned_model = DecoderOnlyLanguageModel(
    vocabulary_size=vocabulary_size,
    seed=SEED,
)

_ = instruction_tuned_model(example_input)

selected_instruction_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
selected_instruction_validation_loss = tf.Variable(np.inf, dtype=tf.float32, trainable=False)

instruction_restore_checkpoint = tf.train.Checkpoint(
    model=instruction_tuned_model,
    best_epoch=selected_instruction_epoch,
    best_validation_loss=selected_instruction_validation_loss,
)

if instruction_best_manager.latest_checkpoint is None:
    raise RuntimeError("Best instruction checkpoint not found.")

instruction_restore_checkpoint.restore(
    instruction_best_manager.latest_checkpoint
).assert_existing_objects_matched()

final_instruction_train_loss = evaluate_masked_loss(
    instruction_tuned_model,
    make_instruction_dataset(
        instruction_train_inputs,
        instruction_train_targets,
        instruction_train_masks,
        INSTRUCTION_BATCH_SIZE,
    ),
)

final_instruction_validation_loss = evaluate_masked_loss(instruction_tuned_model, instruction_validation_dataset)

instruction_test_loss = evaluate_masked_loss(instruction_tuned_model, instruction_test_dataset)

instruction_test_perplexity = float(np.exp(instruction_test_loss))

print("Selected instruction epoch:", int(selected_instruction_epoch.numpy()))
print(f"Final masked training loss: {final_instruction_train_loss:.6f}")
print(f"Final masked validation loss: {final_instruction_validation_loss:.6f}")
print(f"Instruction test masked loss: {instruction_test_loss:.6f}")
print(f"Instruction test perplexity: {instruction_test_perplexity:.3f}")

Selected instruction epoch: 5
Final masked training loss: 4.883737
Final masked validation loss: 5.180870
Instruction test masked loss: 5.182242
Instruction test perplexity: 178.082


## 15. Base and Instruction-Tuned Controlled Generation

Version 13 introduced explicit temperature, top-k and top-p sampling.

Version 14 reused those controls for a base-versus-instruction-tuned comparison.

Version 15 preserves the same comparison so the effect of the new multi-head architecture is not mixed with a new decoding procedure.

Both models receive the same held-out Dolly prompt, generation length and random seed.

Two decoding modes are shown:

- greedy decoding;
- top-p sampling with `top_p = 0.9`.

The context remains only 16 BPE tokens, so the comparison is still an educational demonstration rather than assistant-quality generation.

In [35]:
def controlled_probability_distribution(logits, temperature=1.0, top_k=None, top_p=None):
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero")

    logits = np.asarray(logits, dtype=np.float64)
    scaled_logits = logits / temperature
    scaled_logits -= np.max(scaled_logits)

    probabilities = np.exp(scaled_logits)
    probabilities /= probabilities.sum()

    if top_k is not None:
        if not isinstance(top_k, int) or not 1 <= top_k <= len(probabilities):
            raise ValueError("top_k must be an integer inside the vocabulary range")

        retained_indices = np.argpartition(probabilities, -top_k)[-top_k:]
        filtered = np.zeros_like(probabilities)
        filtered[retained_indices] = probabilities[retained_indices]
        probabilities = filtered / filtered.sum()

    if top_p is not None:
        if not 0 < top_p <= 1:
            raise ValueError("top_p must be in the interval (0, 1]")

        sorted_indices = np.argsort(probabilities)[::-1]
        cumulative = np.cumsum(probabilities[sorted_indices])
        retained_count = int(np.searchsorted(cumulative, top_p, side="left")) + 1
        retained_indices = sorted_indices[:retained_count]

        filtered = np.zeros_like(probabilities)
        filtered[retained_indices] = probabilities[retained_indices]
        probabilities = filtered / filtered.sum()

    return probabilities


def next_token_logits(model, context_ids):
    input_tensor = tf.constant([context_ids], dtype=tf.int32)
    _, _, logits = model(input_tensor)
    return logits[0, -1].numpy()


def generate_text(
    model,
    tokenizer,
    prompt,
    number_of_tokens,
    method="greedy",
    temperature=1.0,
    top_k=None,
    top_p=None,
    seed=42,
):
    generated_ids = tokenizer.encode(prompt).ids

    if len(generated_ids) < CONTEXT_LENGTH:
        raise ValueError(f"The prompt must contain at least {CONTEXT_LENGTH} tokens.")

    random_generator = np.random.default_rng(seed)

    for _ in range(number_of_tokens):
        context_ids = generated_ids[-CONTEXT_LENGTH:]
        logits = next_token_logits(model, context_ids)

        if method == "greedy":
            next_token_id = int(np.argmax(logits))

        elif method == "sample":
            probabilities = controlled_probability_distribution(
                logits,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
            )

            next_token_id = int(
                random_generator.choice(
                    vocabulary_size,
                    p=probabilities,
                )
            )

        else:
            raise ValueError("method must be 'greedy' or 'sample'")

        generated_ids.append(next_token_id)

    return generated_ids, tokenizer.decode(generated_ids)

In [36]:
comparison_example = next(
    row.to_dict()
    for _, row in instruction_test_data.iterrows()
    if len(tokenizer.encode(format_instruction_prompt(row.to_dict())).ids) >= CONTEXT_LENGTH
)

comparison_prompt = format_instruction_prompt(comparison_example)

generation_comparisons = {}

for model_name, comparison_model in [("base", final_model), ("instruction-tuned", instruction_tuned_model)]:
    greedy_ids, greedy_text = generate_text(
        comparison_model,
        tokenizer,
        comparison_prompt,
        INSTRUCTION_GENERATION_TOKENS,
        method="greedy",
        seed=SEED,
    )

    top_p_ids, top_p_text = generate_text(
        comparison_model,
        tokenizer,
        comparison_prompt,
        INSTRUCTION_GENERATION_TOKENS,
        method="sample",
        top_p=0.9,
        seed=SEED,
    )

    generation_comparisons[model_name] = {
        "greedy_ids": greedy_ids,
        "greedy_text": greedy_text,
        "top_p_ids": top_p_ids,
        "top_p_text": top_p_text,
    }

    print(f"\n{model_name} model")
    print("Greedy:")
    print(greedy_text)

    print("\nTop-p 0.9:")
    print(top_p_text)

print("\nReference response:")
print(comparison_example["response"].strip())


base model
Greedy:
Instruction : Tell me why people like the Skate video game series in one sentence Context : The Skate series distinguishes itself from other skateboarding video games with its unique control system , called " Flick - it ", in which players execute tricks with brief , rapid movements of either or both analog sticks of a twin - stick gamepad . Players assume the role of a customizable skater and freely explore a fictional city in an open world setting , completing diverse and increasingly difficult challenges while being filmed by cameraman Giovanni Reda to unlock new areas , obtain new cosmetics and eventually rise to stardom . Along the way , they can encounter , compete with and even play as famous real - life skaters , including Danny Way , Terry Kennedy and Rob Dyrdek , and must eventually choose from among several authentic licensed skateboard gear firms to strike an endorsement with for exclusive skateboard parts and gear . Players can make use of session marke

## 16. Results

Version 15 preserves the complete Version 14 data, training, checkpoint, evaluation and instruction-tuning pipeline while changing one architectural concept: causal self-attention is now explicitly multi-head.

The original 16-dimensional attention projection is divided into two 8-dimensional heads inside each of the two Transformer blocks.

Each head receives the same causal mask but computes its own attention distribution.

The head outputs are concatenated before the unchanged attention-output projection.

Because all projection dimensions remain the same, the model still contains exactly 34,496 trainable parameters.

The Version 15 progression is therefore:

Version 14 experimental pipeline reproduced in Version 15  
→ split 16 attention dimensions into two heads  
→ independent causal attention per head  
→ concatenated head representations  
→ unchanged base training and checkpoint selection  
→ unchanged supervised instruction tuning  
→ unchanged held-out evaluation and controlled generation

The notebook therefore isolates multi-head attention rather than combining it with optimizer, width, depth, dropout or training-schedule changes.

## 17. Tests

The following checks preserve the structural, checkpoint, held-out evaluation and instruction-tuning guarantees of Version 14 and verify the new Version 15 multi-head behavior.

They confirm that the attention dimension is divided exactly across two heads, every block returns the expected per-head attention shape, each head is independently normalized, all future positions remain masked in every head, the parameter count remains unchanged, and the complete base-training and instruction-tuning pipeline still satisfies the previous validation and reproducibility checks.

In [37]:
assert vocabulary_size == VOCABULARY_SIZE
assert tokenizer.token_to_id("[UNK]") is not None

assert train_inputs.shape == (MAX_TRAIN_SEQUENCES, CONTEXT_LENGTH)
assert validation_inputs.shape == (MAX_VALIDATION_SEQUENCES, CONTEXT_LENGTH)
assert test_inputs.shape == (MAX_TEST_SEQUENCES, CONTEXT_LENGTH)

assert total_parameters == 34496
assert len(model.trainable_variables) == 23
assert len(model.blocks) == NUM_BLOCKS

assert NUM_HEADS == 2
assert HEAD_DIM == 8
assert NUM_HEADS * HEAD_DIM == ATTENTION_DIM

for block_attention in attention_weights:
    assert block_attention.shape == (1, NUM_HEADS, CONTEXT_LENGTH, CONTEXT_LENGTH)

assert restored_epoch == INTERRUPT_AFTER_EPOCH
assert int(resumed_completed_epoch.numpy()) == TOTAL_EPOCHS
assert resumed_best_manager.latest_checkpoint is not None

assert np.isfinite(base_test_metrics["perplexity"])
assert 0 <= base_test_metrics["top_1_accuracy"] <= base_test_metrics["top_5_accuracy"] <= 1

assert len(instruction_train_data) == INSTRUCTION_TRAIN_ROWS
assert len(instruction_validation_data) == INSTRUCTION_VALIDATION_ROWS
assert len(instruction_test_data) == INSTRUCTION_TEST_ROWS

instruction_source_sets = [
    set(instruction_train_data["source_index"]),
    set(instruction_validation_data["source_index"]),
    set(instruction_test_data["source_index"]),
]

assert instruction_source_sets[0].isdisjoint(instruction_source_sets[1])
assert instruction_source_sets[0].isdisjoint(instruction_source_sets[2])
assert instruction_source_sets[1].isdisjoint(instruction_source_sets[2])

assert format_instruction_prompt(formatted_example).startswith("Instruction:\n")
assert format_instruction_prompt(formatted_example).endswith("Response:\n")

for inputs, targets, masks in [
    (instruction_train_inputs, instruction_train_targets, instruction_train_masks),
    (instruction_validation_inputs, instruction_validation_targets, instruction_validation_masks),
    (instruction_test_inputs, instruction_test_targets, instruction_test_masks),
]:
    assert inputs.shape == targets.shape == masks.shape
    assert inputs.shape[1] == CONTEXT_LENGTH
    assert np.all(np.isin(masks.numpy(), [0.0, 1.0]))
    assert np.all(masks.numpy().sum(axis=1) > 0)

assert np.any(inspection_mask == 0)
assert np.any(inspection_mask == 1)

assert instruction_start_matches_base
assert instruction_parameters_updated
assert instruction_best_manager.latest_checkpoint is not None
assert 1 <= int(selected_instruction_epoch.numpy()) <= INSTRUCTION_EPOCHS

assert np.all(np.isfinite([
    initial_instruction_train_loss,
    initial_instruction_validation_loss,
    final_instruction_train_loss,
    final_instruction_validation_loss,
    instruction_test_loss,
    instruction_test_perplexity,
]))

assert np.isclose(final_instruction_validation_loss, float(selected_instruction_validation_loss.numpy()), atol=1e-5)

prompt_length = len(tokenizer.encode(comparison_prompt).ids)

for result in generation_comparisons.values():
    assert len(result["greedy_ids"]) == prompt_length + INSTRUCTION_GENERATION_TOKENS
    assert len(result["top_p_ids"]) == prompt_length + INSTRUCTION_GENERATION_TOKENS

repeated_top_p_ids, _ = generate_text(
    instruction_tuned_model,
    tokenizer,
    comparison_prompt,
    INSTRUCTION_GENERATION_TOKENS,
    method="sample",
    top_p=0.9,
    seed=SEED,
)

assert repeated_top_p_ids == generation_comparisons["instruction-tuned"]["top_p_ids"]

_, tuned_attention, _ = instruction_tuned_model(test_inputs[0:1])

for block_attention in tuned_attention:
    matrix = block_attention[0].numpy()
    assert matrix.shape == (NUM_HEADS, CONTEXT_LENGTH, CONTEXT_LENGTH)
    assert np.allclose(matrix.sum(axis=-1), 1.0, atol=1e-6)

    for head_matrix in matrix:
        assert np.allclose(np.triu(head_matrix, k=1), 0.0, atol=1e-7)

print("All checks passed.")

All checks passed.


## Notes

- Version 15 is Version 14 plus explicit multi-head causal self-attention.

- WikiText-103 Raw v1, BPE tokenization, sequence limits, model width, feed-forward width, two-block depth, base checkpoint behavior, learning rate, batch size and 20-epoch base-training budget remain unchanged.

- `ATTENTION_DIM = 16` is now divided into two independent heads of eight dimensions each.

- Query, key, value and attention-output projection shapes remain unchanged, so the model still contains exactly 34,496 trainable parameters.

- Every head receives the same causal mask but computes its own normalized attention distribution.

- The held-out WikiText evaluation and controlled-generation procedure introduced in Version 13 are preserved.

- The Dolly supervised instruction-tuning stage introduced in Version 14 is also preserved, including the same 192/32/32 split, response-focused masked loss and validation-selected instruction checkpoint.

- The BPE tokenizer is still trained only on WikiText training text and is not retrained on Dolly.

- Base checkpointing and instruction checkpointing remain intentionally minimal. Version 15 does not add Adam, learning-rate scheduling, gradient clipping, dropout, wider embeddings, deeper blocks, persistent cross-session resume or checkpoint contracts.

- The context length remains only 16 BPE tokens, so long instructions are truncated during generation and outputs remain limited by the deliberately small model.

Future versions will introduce new components and gradually evolve the architecture.